Script for building FreeSurfer 'recon-all' commands and piping them out to the terminal for MRI volumetric reconstruction.

[Runtime: ~2-10 **hours** per subject (not *file*) in dataset, depending on local machine tech specs]

---------

In [ ]:
### LOAD CONFIG.YAML:

import yaml, os
from pathlib import Path

CONFIG_PATH = Path.cwd() / "config.yaml"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Config file not found: {CONFIG_PATH.resolve()}")

with open(CONFIG_PATH, 'r') as config_file:
    config = yaml.safe_load(config_file)

# config

In [ ]:
### IMPORTS:
import os
import datetime
import pandas as pd
import numpy as np
from collections import defaultdict
import difflib
import subprocess
import shutil
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import shutil

In [ ]:
### LOAD FREESURFER:
FREESURFER_HOME = config['freesurfer']['home']
FREESURFER_LICENSE = Path(config['freesurfer'].get('license'))
FREESURFER_SUBJECTS_DIR = config['freesurfer']['subjects_dir']
os.environ['FREESURFER_HOME'] = str(FREESURFER_HOME)
os.environ['SUBJECTS_DIR']    = str(FREESURFER_SUBJECTS_DIR)
os.environ['PATH'] = f"{str(FREESURFER_HOME)}/bin:" + os.environ.get('PATH', '')
if FREESURFER_LICENSE.is_file():
    os.environ['FS_LICENSE'] = str(FREESURFER_LICENSE)
else:
    raise RuntimeError("FreeSurfer license not found: check that 'license.txt' exists in home FreeSurfer directory."
    "in FreeSurfer $HOME directory and that a valid path is set in 'config.yaml' file.")
subprocess.run("recon-all --version", shell=True, check=True)

In [ ]:
### SET PARAMETERS:
HARD_STOP = config['hard_errors']
DRY_RUN = config['dry_run']
SUBSET = config['subset']

### SET PATHS:
ROOT_DIR = Path(config['root_output_directory'])

# INPUTS:
MRI_DATA_DIR = config['MRI_data_directory']

RUN_MANIFEST_PATH = Path(ROOT_DIR) / 'subject_manifest.csv'
RUN_MANIFEST = pd.read_csv(RUN_MANIFEST_PATH)

fMRI_MANIFEST_PATH = Path(ROOT_DIR) / 'fMRI_manifest.csv' 
fMRI_runs = pd.read_csv(fMRI_MANIFEST_PATH)

**Optional subsetting:**

- Note: We use a different approach here, because we always process only 1 MRI file per subject, but may be planning on analyzing more than one fMRI per subject; so if we are subsetting to only perform an analysis run of the first say '10' fMRI files, with 2 per subject, we only need to run 'recon-all' for 5 subject_IDs in total in order to have all the anatomical data we need. This is important to "optimize" since this initial 'recon-all' step is by far the most time- and compute-intensive stage of the entire pipeline, and we don't want to run it on more subject_IDs than is necessary for a given subset size.
- So in practice we subset on the fMRI file manifest, grab the unique subject_IDs selected, then constrain the MRI manifest (here the same as RUN_MANIFEST) by that same set of subject_IDs:

In [ ]:
### DIAGNOSTIC SUBSETTING (if enabled):
if isinstance(SUBSET, int) and SUBSET > 0:
    print(f"\nDIAGNOSTIC SUBSETTING ENABLED: Selecting all subject_IDs from first {SUBSET} available fMRI files:")
    subset_subjects = (
        fMRI_runs
        .head(SUBSET)
        .loc[:, "subject_ID"]
        .dropna()
        .astype(str)
        .unique())
    before_n = len(RUN_MANIFEST)
    RUN_MANIFEST = (
        RUN_MANIFEST[RUN_MANIFEST["subject_ID"].astype(str).isin(subset_subjects)].copy())
    after_n = len(RUN_MANIFEST)
    print(f"  → Subsetting: {before_n} → {after_n}")
    print(f"  → MRI volume reconstructions will be performed for {after_n} subject_IDs.")
    display(RUN_MANIFEST)

For simplicity's sake we'll discard any fMRI-related columns from the run manifest for now:

In [ ]:
MRI_runs = RUN_MANIFEST.loc[:, [column for column in RUN_MANIFEST.columns if 'fMRI' not in column]]

Let's do a recursive search in the MRI root directory for all the files included in our run manifest:

In [ ]:
# =========================
# Resolve MRI filepaths (.nii preferred over .nii.gz); warn on unexpected multiples/duplicates
# =========================

base_dir = Path(MRI_DATA_DIR)
if not base_dir.exists():
    raise FileNotFoundError(f"[error] MRI_DATA_DIR does not exist: {base_dir}")

MRI_runs = MRI_runs.copy()
MRI_runs['MRI_filepath'] = np.nan

n_found = 0
n_missing = 0
n_multi_warn = 0

for idx, row in MRI_runs.iterrows():
    subj = str(row['subject_ID'])
    stem = str(row['MRI_filename'])

    # Find matching files ('.nii' or '.nii.gz'):
    hits = [p.resolve() for p in base_dir.rglob(stem + '.nii') if p.is_file()]
    hits += [p.resolve() for p in base_dir.rglob(stem + '.nii.gz') if p.is_file()]
    hits = list(dict.fromkeys(hits))  # <-- de-duplicates

    n_matches = len(hits)

    if n_matches == 0:
        n_missing += 1
        continue

    # Prefer .nii over .nii.gz when both exist:
    if n_matches == 2:
        has_nii = any(str(h).endswith('.nii') for h in hits)
        has_niigz = any(str(h).endswith('.nii.gz') for h in hits)
        if has_nii and has_niigz:
            chosen = [h for h in hits if str(h).endswith('.nii')][0]
            MRI_runs.at[idx, 'MRI_filepath'] = str(chosen)
            n_found += 1
            continue

    # Otherwise, pick .nii if present, else first hit:
    chosen = next((h for h in hits if str(h).endswith('.nii')), hits[0])
    MRI_runs.at[idx, 'MRI_filepath'] = str(chosen)
    n_found += 1

    # Warn on unexpected multiplicity:
    if n_matches > 1 and not (n_matches == 2 and has_nii and has_niigz):
        n_multi_warn += 1
        print(f"[warn] Multiple hits for subject_ID={subj}, MRI_filename={stem}:")
        for h in hits:
            print(f"       - {h}")
        print(f"       -> Selected: {chosen}")

# Print summary:
total = len(MRI_runs)
print(f"[scan] ROOT = {base_dir}")
print(f"[match] Filled paths for {n_found}; missing {n_missing}; multiple matches {n_multi_warn}")

if n_missing > 0:
    print("[warn] Missing examples (up to 5):")
    print(MRI_runs[MRI_runs['MRI_filepath'].isna()]
          [['subject_ID', 'MRI_filename']]
          .head(5)
          .to_string(index=False))

In [ ]:
assert MRI_runs.MRI_filepath.isna().sum() == 0

----------
Re-implementing old code, with modifications for input routing only:

In [ ]:
# Set compute-parallelization settings:
def _pick_parallel_params(openmp_threads=None, max_workers=None):
    ncpu = os.cpu_count() or 8
    if openmp_threads is None:
        openmp_threads = max(1, min(8, ncpu // 2))
    if max_workers is None:
        max_workers = max(1, ncpu // openmp_threads)
    return openmp_threads, max_workers, ncpu

# build the 'recon-all' command:
def _build_recon_cmd(in_nii, subj_id, subjects_dir, openmp_threads):
    return [
        "recon-all",
        "-i", str(in_nii),
        "-s", str(subj_id),
        "-sd", str(subjects_dir),
        "-all",
        "-parallel",
        "-openmp", str(openmp_threads)]

# FreeSurfer 'recon-all' batch launcher -- now expects columns ['subject_ID', 'MRI_filepath']
def run_recon_all_batch(
    df,  # <--- can be MRI_runs directly
    subjects_dir=None,
    openmp_threads=None,
    max_workers=None,
    dry_run=False,
    skip_existing=True,
    skip_if_done=True):
    """
    Launch 'recon-all' for each row (requires: 'subject_ID','MRI_filepath').

    Behavior preserved from the validated pipeline:
    - No subject dir pre-creation (recon-all creates it).
    - Logs written to <SUBJECTS_DIR>/_launcher_logs/*.log
    - Skip if recon-all.done exists or subject dir exists (configurable).
    - DRY_RUN prints commands for previewing only.
    """
    if subjects_dir is None:
        subjects_dir = os.environ.get("SUBJECTS_DIR")
    if not subjects_dir:
        raise EnvironmentError("SUBJECTS_DIR is not set.")
    if shutil.which("recon-all") is None:
        raise FileNotFoundError("Could not find 'recon-all' in PATH.")

    required_cols = {"subject_ID", "MRI_filepath"}
    missing = required_cols - set(df.columns)
    if missing:
        raise KeyError(f"Input dataframe missing required columns: {sorted(missing)}")

    # Parallel params + logging root:
    openmp_threads, max_workers, ncpu = _pick_parallel_params(openmp_threads, max_workers)
    print(f"[INFO] CPU cores: {ncpu} | per-subject threads: {openmp_threads} | concurrent subjects: {max_workers}")
    print(f"[INFO] SUBJECTS_DIR = {subjects_dir}")
    if dry_run:
        print("[INFO] DRY-RUN mode: no subject dirs/logs will be created under SUBJECTS_DIR.\n")

    # Only create directories/log roots when not in DRY_RUN:
    if not dry_run:
        os.makedirs(subjects_dir, exist_ok=True)
        logs_root = Path(subjects_dir) / "_launcher_logs"
        logs_root.mkdir(parents=True, exist_ok=True)
    else:
        logs_root = Path(subjects_dir) / "_launcher_logs"  # <-- for consistent path formatting only

    failures = []

    def _run_one(subj_id, in_nii):
        subj_dir = Path(subjects_dir) / subj_id
        done_flag = subj_dir / "scripts" / "recon-all.done"
        ext_runtime_log = logs_root / f"{subj_id}.runtime_log.txt"
        ext_proc_log    = logs_root / f"{subj_id}.recon-all.stdout.log"

        # Skip logic BEFORE any side effects:
        if skip_if_done and done_flag.exists():
            print(f"[SKIP-DONE] {subj_id} (recon-all.done present)")
            return subj_id, 0, None
        # Avoid silently skipping partial/failed subjects:
        # Only skip on "existing dir" if it is actually done, unless user explicitly disables skip_if_done:
        if skip_existing and subj_dir.exists() and not dry_run:
            if done_flag.exists():
                print(f"[SKIP-EXISTS] {subj_id} (subject directory exists and recon-all.done present)")
                return subj_id, 0, None
            else:
                print(f"[WARN] {subj_id} subject directory exists but recon-all.done is missing → treating as incomplete; will re-run.")

        cmd = _build_recon_cmd(in_nii, subj_id, subjects_dir, openmp_threads)
        print(f"[START]{' [DRY] ' if dry_run else ' '}{subj_id} | {in_nii}\n  CMD: {' '.join(cmd)}")

        # 'DRY_RUN' mode only prints constructed commands for user review:
        if dry_run:
            return subj_id, 0, None

        start = datetime.datetime.now()
        with open(ext_runtime_log, "a") as lf:
            lf.write(f"Date: {start.strftime('%Y-%m-%d')}\n")
            lf.write(f"Start time: {start.strftime('%H:%M:%S')}\n")
            lf.write(f"Command: {' '.join(cmd)}\n")

        rc = 0
        err_msg = None
        env = os.environ.copy()
        env["SUBJECTS_DIR"] = str(subjects_dir)
        try:
            with open(ext_proc_log, "a") as pout:
                proc = subprocess.run(cmd, env=env, stdout=pout, stderr=subprocess.STDOUT, text=True)
                rc = proc.returncode
                if rc != 0:
                    err_msg = f"Non-zero exit ({rc}). See log: {ext_proc_log}"
        except Exception as e:
            rc = -1
            err_msg = f"Exception: {e}"

        end = datetime.datetime.now()
        duration = str(end - start)
        with open(ext_runtime_log, "a") as lf:
            lf.write(f"End time: {end.strftime('%H:%M:%S')}\n")
            lf.write(f"Total runtime: {duration}\n")
            if err_msg:
                lf.write(f"ERROR: {err_msg}\n")
            lf.write("-" * 40 + "\n")

        if subj_dir.exists():
            try:
                with open(subj_dir / "runtime_log.txt", "a") as lf:
                    lf.write(f"Date: {start.strftime('%Y-%m-%d')}\n")
                    lf.write(f"Start time: {start.strftime('%H:%M:%S')}\n")
                    lf.write(f"End time: {end.strftime('%H:%M:%S')}\n")
                    lf.write(f"Total runtime: {duration}\n")
                    if err_msg:
                        lf.write(f"ERROR: {err_msg}\n")
                    lf.write("-" * 40 + "\n")
            except Exception:
                pass

        if err_msg:
            print(f"[FAIL]  {subj_id} | {err_msg}")
        else:
            print(f"[DONE]  {subj_id} | {duration}")

        return subj_id, rc, err_msg

    # Parallel execution:
    from concurrent.futures import as_completed
    futures = []
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        for _, r in df.iterrows():
            sid = str(r['subject_ID'])
            nii = str(r['MRI_filepath'])   # <--- minimal change here
            futures.append(ex.submit(_run_one, sid, nii))

        for fut in as_completed(futures):
            sid, rc, err = fut.result()
            if rc != 0:
                failures.append((sid, rc, err))

    if failures:
        print(f"\n[SUMMARY] {len(failures)} subject(s) failed:")
        for sid, rc, err in failures[:20]:
            print(f"  - {sid}: rc={rc} | {err}")
        if len(failures) > 20:
            print(f"  ... and {len(failures)-20} more")
    else:
        print("\n[SUMMARY] All subjects completed successfully (or were skipped).")

    return {"failures": failures, "threads": openmp_threads, "workers": max_workers}

Actual execution code:

In [ ]:
run_recon_all_batch(
    MRI_runs,
    subjects_dir=os.environ["SUBJECTS_DIR"],
    skip_existing=True,
    skip_if_done=True,
    dry_run=DRY_RUN)